In [1]:
import joblib
import h5py
import numpy as np
from utils import save, load
from library import *
import gc
from continuous_process_library_terms import *
from commons_identify_models import *
import copy
import jax
import jax.numpy as jnp
from jax_cfd.base import boundaries, grids, pressure, finite_differences as fd
import jax_cfd.base as cfd
import jax_cfd.base.grids as grids
import jax_cfd.spectral as spectral
import numpy as np
#import xarray 
#import xarray as xr
import dataclasses
import time
jax.config.update("jax_enable_x64", True)
import numpy as np
import os, joblib
from joblib import Parallel, delayed
from tqdm.auto import tqdm
from numpy.polynomial.chebyshev import chebvander

import contextlib
import joblib
from tqdm import tqdm

@contextlib.contextmanager
def tqdm_joblib(tqdm_object):
    """Context manager to patch joblib to report into tqdm progress bar given as argument"""
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old_batch_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_batch_callback
        tqdm_object.close()

/storage/home/hcoda1/2/ashelestov3/.conda/envs/SPIDER/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

f = h5py.File('/storage/scratch1/2/ashelestov3/check_paper_stabilized/check_paper_stabilized_s1.h5', 'r')
# '/storage/scratch1/2/ashelestov3/snaps_1024_1e8/snaps_1024_1e8_s1.h5'
# '/storage/scratch1/2/ashelestov3/snaps_2048_1e8/snaps_2048_1e8_s1.h5'
print(list(f['tasks'].keys()))
'''
f['tasks']['buoyancy'] = f['buoyancy'][901:]
f['tasks']['pressure'] = f['pressure'][901:]
f['tasks']['velocity'] = f['velocity'][901:]
f['tasks']['vorticity'] = f['vorticity'][901:]
'''
scales=f['scales']
print(list(scales.keys()))
f = f['tasks']
list(f.keys())



['b', 'p', 'tau_b1', 'tau_b2', 'tau_p', 'tau_u1', 'tau_u2', 'u']
['constant', 'iteration', 'sim_time', 'timestep', 'wall_time', 'write_number', 'x_hash_4846e26f3f50e48ca5da4b8e0970d850261c704a', 'z_hash_050b2c5be05fb65046d37e4f71ee7ab1962cccdc']


['b', 'p', 'tau_b1', 'tau_b2', 'tau_p', 'tau_u1', 'tau_u2', 'u']

In [3]:
print(f['u'].shape)

(1, 2, 4096, 1536)


In [4]:
import numpy as np
u=f['u']

# 1. Setup the Grid
# ---------------------------------------------------------
Ny = 4096
Nx = 1536
Lx = 4  # CHANGE THIS: Total length of your x-domain (e.g., 4, 2*pi, etc.)
Ly_min, Ly_max = scales['z_hash_050b2c5be05fb65046d37e4f71ee7ab1962cccdc'][0],  scales['z_hash_050b2c5be05fb65046d37e4f71ee7ab1962cccdc'][-1] # CHANGE THIS: Bounds of your y-domain

# Generate Chebyshev Grid (Gauss-Lobatto nodes are standard for Dedalus)
# Formula: y_j = -cos(j * pi / (Ny - 1)) scaled to [min, max]
n = np.arange(Ny)
y_cheb = -np.cos(n * np.pi / (Ny - 1)) 
# Rescale y_cheb to physical domain if not [-1, 1]
y_phys = Ly_min + (Ly_max - Ly_min) * (y_cheb + 1) / 2.0

# Define dx (Uniform)
dx = Lx / Nx

# 2. Extract Data
# ---------------------------------------------------------
# Assuming u has shape (1, 2, 4096, 1536)
# Squeeze to remove time dimension -> (2, 4096, 1536)
u_data = np.squeeze(u) 

# Extract components (Modify based on your actual component order)
# Assuming index 0 is Y-velocity, index 1 is X-velocity
uy = u_data[0, :, :] 
ux = u_data[1, :, :]

# 3. Calculate Derivatives
# ---------------------------------------------------------
# d(uy)/dx: Uniform grid, use scalar dx
duy_dx = np.gradient(uy, dx, axis=1)

# d(ux)/dy: Non-uniform Chebyshev grid, pass the y_phys array
# np.gradient handles the non-uniform spacing automatically
dux_dy = np.gradient(ux, y_phys, axis=0)

# 4. Calculate Vorticity
# ---------------------------------------------------------
# Vorticity = dv/dx - du/dy (or du_y/dx - du_x/dy)
vorticity = duy_dx - dux_dy

# 5. Calculate Eddy Turnover Time
# ---------------------------------------------------------
# Standard robust calculation (Enstrophy based)
rms_vorticity = np.sqrt(np.mean(vorticity**2))
turnover_time = 1 / rms_vorticity

print(f"Eddy Turnover Time (RMS): {turnover_time:.4e}")

# Your requested calculation (Warning: Likely Inf if flow is symmetric)
try:
    print(f"Requested Mean Time: {1/np.mean(vorticity):.4e}")
except ZeroDivisionError:
    print("Mean vorticity is zero (infinite time).")

Eddy Turnover Time (RMS): 2.4039e-03
Requested Mean Time: 4.9038e-02
